# Día 5 — Taller Integrador: ENIGH 2022 + Programas Sociales
### Curso: Introducción a Herramientas de Cómputo en la Nube · Azure

---

**Pregunta analítica del día:**
> *¿Cuál es el perfil socioeconómico de los hogares mexicanos que reciben transferencias de programas sociales del gobierno?*

**Datasets:**
- **ENIGH 2022** (INEGI) — microdatos de hogares: ingreso, gasto, características sociodemográficas
- **Programas sociales sintéticos** — estructura del padrón de beneficiarios (Sembrando Vida, Pensión Bienestar, Becas Benito Juárez)

**Servicios de Azure que usamos hoy:**
- Blob Storage — almacenar los datos descargados
- PostgreSQL + PostGIS — persistir resultados limpios
- Azure ML — lanzar un Command Job de clasificación

| Sección | Contenido | Servicio |
|---------|-----------|----------|
| 0 | Instalación y credenciales | — |
| 1 | Descargar ENIGH 2022 y subir a Blob | Blob Storage |
| 2 | Limpiar y explorar la ENIGH | pandas |
| 3 | Generar datos de programas sociales | pandas |
| 4 | Cruzar ENIGH con programas sociales | pandas |
| 5 | Análisis del perfil: visualizaciones | matplotlib |
| 6 | Guardar resultados en PostgreSQL | psycopg2 |
| 7 | Modelo de clasificación en Azure ML | azure-ai-ml |
| 8 | Cierre: resumen del curso | — |

---
## Sección 0 — Instalación y credenciales

In [1]:
%pip install adlfs fsspec psycopg2-binary azure-ai-ml azure-identity \
             pandas matplotlib seaborn requests python-dotenv --quiet
print('Listo.')

Note: you may need to restart the kernel to use updated packages.
Listo.


In [2]:
# Las credenciales se leen desde credenciales.env
# generado automaticamente por setup_dia5.sh
#
# NUNCA pongas valores directamente aqui — ese archivo va a .gitignore
#
# Si no tienes el .env todavia, crea credenciales.env con:
#   STORAGE_ACCOUNT_NAME=tu_cuenta
#   STORAGE_ACCOUNT_KEY=tu_clave
#   BLOB_CONTAINER=enigh-datos
#   PG_HOST=tu-servidor.postgres.database.azure.com
#   PG_PORT=5432
#   PG_DATABASE=cursodb
#   PG_USER=cursoazure
#   PG_PASSWORD=tu_password
#   AML_SUBSCRIPTION=tu_subscription_id
#   AML_RESOURCE_GROUP=tu_resource_group
#   AML_WORKSPACE=aml-workspace-dia5

import os
from dotenv import load_dotenv

load_dotenv('credenciales.env')

# ── Blob Storage ──────────────────────────────────────────────────────
STORAGE_ACCOUNT_NAME = os.environ['STORAGE_ACCOUNT_NAME']
STORAGE_ACCOUNT_KEY  = os.environ['STORAGE_ACCOUNT_KEY']
BLOB_CONTAINER       = os.environ.get('BLOB_CONTAINER', 'enigh-datos')

# ── PostgreSQL ────────────────────────────────────────────────────────
PG_HOST     = os.environ['PG_HOST']
PG_PORT     = int(os.environ.get('PG_PORT', 5432))
PG_DATABASE = os.environ.get('PG_DATABASE', 'cursodb')
PG_USER     = os.environ['PG_USER']
PG_PASSWORD = os.environ['PG_PASSWORD']

# ── Azure ML ──────────────────────────────────────────────────────────
#AML_SUBSCRIPTION   = os.environ['AML_SUBSCRIPTION']
#AML_RESOURCE_GROUP = os.environ['AML_RESOURCE_GROUP']
AML_WORKSPACE      = os.environ.get('AML_WORKSPACE', 'aml-workspace-dia5')

print('Credenciales cargadas desde credenciales.env')
print(f'  Storage : {STORAGE_ACCOUNT_NAME}')
print(f'  PG host : {PG_HOST}')
print(f'  AML     : {AML_WORKSPACE}')

Credenciales cargadas desde credenciales.env
  Storage : sargtallerdia5
  PG host : pg-rgtallerdia5-dia5.postgres.database.azure.com
  AML     : aml-workspace-dia5


---
## Sección 1 — Descargar ENIGH 2022 y subir a Blob Storage

La ENIGH 2022 está disponible en el servidor de microdatos del INEGI como ZIP.  
Descargamos `hogares.csv` directamente en memoria y lo subimos a Blob sin guardar en disco.

**Variables que usaremos de hogares.csv:**
- `folioviv`, `foliohog` — identificadores únicos del hogar
- `ubica_geo` — clave geográfica (primeros 2 dígitos = entidad)
- `tam_loc` — tamaño de localidad (1=+100k hab, 2=15-100k, 3=2.5-15k, 4=rural <2.5k)
- `ing_cor` — ingreso corriente total trimestral (pesos)
- `gasto_mon` — gasto monetario total trimestral
- `alimentos` — gasto en alimentos
- `salud` — gasto en salud
- `educa_espa` — gasto en educación
- `transporte` — gasto en transporte
- `bene_gob` — transferencias recibidas del gobierno (programas sociales)
- `factor` — factor de expansión muestral

In [3]:
import requests, zipfile, io, pandas as pd

# La ENIGH 2022 distribuye los datos en varios archivos:
#   hogares.csv           -> caracteristicas del hogar (huespedes, acc. alimentos)
#   concentradohogar.csv  -> ingreso, gasto, transferencias  <-- este necesitamos
#   poblacion.csv         -> datos de cada integrante
#   gastoshogar.csv       -> detalle de cada gasto

URL_ENIGH = 'https://www.inegi.org.mx/contenidos/programas/enigh/nc/2022/microdatos/enigh2022_ns_concentradohogar_csv.zip'

print('Descargando ENIGH 2022 concentradohogar...')
print(f'  URL: {URL_ENIGH}')

resp = requests.get(URL_ENIGH, timeout=120)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    archivos = z.namelist()
    print(f'  Archivos en el ZIP: {archivos}')
    nombre_csv = [f for f in archivos if f.endswith('.csv')][0]
    with z.open(nombre_csv) as f:
        # utf-8-sig elimina el BOM que agrega Excel (aparece como ï»¿ en utf-8 normal)
        df_enigh = pd.read_csv(f, encoding='utf-8-sig', low_memory=False)

print(f'\nENIGH 2022 concentradohogar cargada:')
print(f'  Filas    : {len(df_enigh):,}')
print(f'  Columnas : {len(df_enigh.columns)}')
print(f'  Primeras columnas: {list(df_enigh.columns[:10])}')
df_enigh.head(3)


Descargando ENIGH 2022 concentradohogar...
  URL: https://www.inegi.org.mx/contenidos/programas/enigh/nc/2022/microdatos/enigh2022_ns_concentradohogar_csv.zip
  Archivos en el ZIP: ['concentradohogar.csv']

ENIGH 2022 concentradohogar cargada:
  Filas    : 90,102
  Columnas : 126
  Primeras columnas: ['folioviv', 'foliohog', 'ubica_geo', 'tam_loc', 'est_socio', 'est_dis', 'upm', 'factor', 'clase_hog', 'sexo_jefe']


,folioviv,foliohog,ubica_geo,tam_loc,est_socio,est_dis,upm,factor,clase_hog,sexo_jefe,...,mater_serv,material,servicio,deposito,prest_terc,pago_tarje,deudas,balance,otras_erog,smg
0,100005002,1,1001,1,4,3,1,206,3,2,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,15558.3
1,100005003,1,1001,1,4,3,1,206,2,1,...,0.0,0.0,0.0,19565.21,0.0,0.0,0.0,0.0,0.0,15558.3
2,100005004,1,1001,1,4,3,1,206,2,1,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,15558.3


In [4]:
import adlfs

# Crear filesystem fsspec para Azure Blob
fs = adlfs.AzureBlobFileSystem(
    account_name = STORAGE_ACCOUNT_NAME,
    account_key  = STORAGE_ACCOUNT_KEY
)

# Subir hogares.csv a Blob (escribir desde buffer en memoria)
ruta_blob = f'{BLOB_CONTAINER}/enigh2022_concentradohogar.csv'

print(f'Subiendo a Blob: {ruta_blob}...')
with fs.open(ruta_blob, 'wb') as f:
    f.write(df_enigh.to_csv(index=False).encode('utf-8'))

# Verificar
info = fs.info(ruta_blob)
print(f'  Archivo en Blob: {ruta_blob}')
print(f'  Tamaño         : {info["size"] / 1024 / 1024:.1f} MB')
print(f'  Filas subidas  : {len(df_enigh):,}')

Subiendo a Blob: enigh-datos/enigh2022_concentradohogar.csv...
  Archivo en Blob: enigh-datos/enigh2022_concentradohogar.csv
  Tamaño         : 53.8 MB
  Filas subidas  : 90,102


---
## Sección 2 — Limpiar y explorar la ENIGH

Seleccionamos las columnas relevantes, creamos variables derivadas
y hacemos una primera exploración del ingreso y gasto por entidad.

In [5]:
# Columnas de concentradohogar.csv — ENIGH 2022
# Diccionario oficial: https://www.inegi.org.mx/contenidos/programas/enigh/nc/2022/doc/enigh2022_ns_descripcion_de_archivos_y_variables.pdf
COLS = [
    'folioviv', 'foliohog', 'ubica_geo', 'tam_loc',
    'ing_cor',  'gasto_mon', 'alimentos', 'salud',
    'educa_espa', 'transporte', 'bene_gob', 'factor'
]

cols_disponibles = [c for c in COLS if c in df_enigh.columns]
cols_faltantes   = [c for c in COLS if c not in df_enigh.columns]

if cols_faltantes:
    print(f'Columnas no encontradas: {cols_faltantes}')
    print(f'Columnas del archivo descargado:')
    print(list(df_enigh.columns))
    raise ValueError('Archivo incorrecto o version diferente. Revisa el diccionario de variables.')

df = df_enigh[cols_disponibles].copy()

# Clave de entidad federativa (primeros 2 digitos de ubica_geo)
df['entidad'] = df['ubica_geo'].astype(str).str.zfill(5).str[:2].astype(int)

ENTIDADES = {
    1:'Aguascalientes', 2:'Baja California', 3:'Baja California Sur',
    4:'Campeche', 5:'Coahuila', 6:'Colima', 7:'Chiapas', 8:'Chihuahua',
    9:'CDMX', 10:'Durango', 11:'Guanajuato', 12:'Guerrero', 13:'Hidalgo',
    14:'Jalisco', 15:'Estado de Mexico', 16:'Michoacan', 17:'Morelos',
    18:'Nayarit', 19:'Nuevo Leon', 20:'Oaxaca', 21:'Puebla', 22:'Queretaro',
    23:'Quintana Roo', 24:'San Luis Potosi', 25:'Sinaloa', 26:'Sonora',
    27:'Tabasco', 28:'Tamaulipas', 29:'Tlaxcala', 30:'Veracruz',
    31:'Yucatan', 32:'Zacatecas'
}
df['entidad_nom'] = df['entidad'].map(ENTIDADES)

df['tipo_loc'] = df['tam_loc'].map({
    1:'Urbano grande (>100k)', 2:'Urbano medio (15-100k)',
    3:'Urbano pequeno (2.5-15k)', 4:'Rural (<2.5k)'
})

df['recibe_programa'] = (df['bene_gob'] > 0).astype(int)
df['decil_ing'] = pd.qcut(df['ing_cor'], q=10, labels=range(1,11), duplicates='drop')

print(f'Dataset limpio: {len(df):,} hogares')
print(f'  Reciben programa social : {df["recibe_programa"].sum():,} ({df["recibe_programa"].mean()*100:.1f}%)')
print(f'  Ingreso promedio trim.  : ${df["ing_cor"].mean():,.0f} pesos')
print(f'  Entidades representadas : {df["entidad"].nunique()}')
df[['ing_cor','gasto_mon','bene_gob','recibe_programa','tipo_loc']].describe().round(0)


Dataset limpio: 90,102 hogares
  Reciben programa social : 32,039 (35.6%)
  Ingreso promedio trim.  : $61,490 pesos
  Entidades representadas : 32


,ing_cor,gasto_mon,bene_gob,recibe_programa
count,90102.0,90102.0,90102.0,90102.0
mean,61490.0,37615.0,1849.0,0.0
std,78325.0,34649.0,3367.0,0.0
min,0.0,0.0,0.0,0.0
25%,28386.0,18561.0,0.0,0.0
50%,46074.0,29679.0,0.0,0.0
75%,74344.0,45901.0,2611.0,1.0
max,7153770.0,1703575.0,63196.0,1.0


---
## Sección 3 — Generar datos de programas sociales

Creamos un dataset sintético con la estructura del padrón de beneficiarios
(Sembrando Vida, Pensión Bienestar, Becas Benito Juárez).
Los hogares de la ENIGH que tienen `bene_gob > 0` se cruzarán con este padrón.

In [ ]:
import numpy as np
np.random.seed(42)

# Hogares que reportan recibir algún beneficio de gobierno
df_beneficiarios = df[df['recibe_programa'] == 1].copy()
n = len(df_beneficiarios)

# Asignar programa según perfil probable:
# - Sembrando Vida: hogares rurales con ingreso bajo
# - Pensión Bienestar: distribución amplia, peso en adultos mayores
# - Becas Benito Juárez: hogares con niños, ingreso bajo-medio
def asignar_programa(row):
    if row['tipo_loc'] == 'Rural (<2.5k)' and row['ing_cor'] < 15000:
        return np.random.choice(
            ['Sembrando Vida', 'Pensión Bienestar', 'Becas Benito Juárez'],
            p=[0.45, 0.35, 0.20]
        )
    elif row['ing_cor'] < 30000:
        return np.random.choice(
            ['Pensión Bienestar', 'Becas Benito Juárez', 'Sembrando Vida'],
            p=[0.50, 0.35, 0.15]
        )
    else:
        return np.random.choice(
            ['Pensión Bienestar', 'Becas Benito Juárez'],
            p=[0.65, 0.35]
        )

df_beneficiarios = df_beneficiarios.copy()
df_beneficiarios['programa'] = df_beneficiarios.apply(asignar_programa, axis=1)
df_beneficiarios['monto_bimestral'] = df_beneficiarios['bene_gob'] / 1.5  # aprox bimestral

print(f'Hogares beneficiarios: {len(df_beneficiarios):,}')
print()
print('Distribución por programa:')
print(df_beneficiarios['programa'].value_counts().to_string())

---
## Sección 4 — Cruzar ENIGH con programas sociales

Construimos el dataset analítico central: comparar el perfil socioeconómico
de hogares que reciben vs. no reciben programas sociales.

In [ ]:
# Perfil comparativo: beneficiarios vs no beneficiarios
vars_perfil = ['ing_cor', 'gasto_mon', 'alimentos', 'salud', 'educa_espa', 'transporte']
vars_disponibles = [v for v in vars_perfil if v in df.columns]

perfil = df.groupby('recibe_programa')[vars_disponibles].agg(['mean','median']).round(0)
perfil.index = ['No recibe programa', 'Recibe programa']

print('Perfil comparativo (trimestral, pesos):')
print(perfil.to_string())

In [ ]:
# Distribución por tipo de localidad
dist_loc = pd.crosstab(
    df['tipo_loc'],
    df['recibe_programa'],
    normalize='index'
).round(3) * 100
dist_loc.columns = ['% No recibe', '% Recibe']

print('% de hogares que reciben programa por tipo de localidad:')
print(dist_loc.sort_values('% Recibe', ascending=False).to_string())

In [ ]:
# Cobertura de programas por entidad federativa
cobertura_entidad = (
    df.groupby('entidad_nom')['recibe_programa']
    .agg(total='count', beneficiarios='sum')
)
cobertura_entidad['pct_cobertura'] = (
    cobertura_entidad['beneficiarios'] / cobertura_entidad['total'] * 100
).round(1)
cobertura_entidad['ing_mediano'] = (
    df.groupby('entidad_nom')['ing_cor'].median().round(0)
)

print('Cobertura de programas sociales por entidad federativa:')
print(cobertura_entidad.sort_values('pct_cobertura', ascending=False).head(15).to_string())

---
## Sección 5 — Visualizaciones del perfil socioeconómico

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams.update({
    'figure.facecolor': '#0a0e1a',
    'axes.facecolor':   '#111827',
    'axes.edgecolor':   '#1e2d45',
    'axes.labelcolor':  '#6b7a99',
    'xtick.color':      '#6b7a99',
    'ytick.color':      '#6b7a99',
    'text.color':       '#e8edf5',
    'grid.color':       '#1e2d45',
    'grid.linestyle':   '--',
    'font.family':      'monospace'
})

COLORES = {'No recibe': '#00c8ff', 'Recibe': '#00e5a0'}

# ── Figura 1: distribución de ingreso por grupo ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Perfil socioeconómico: beneficiarios vs no beneficiarios\nENIGH 2022 — Hogares mexicanos',
             fontsize=13, y=1.02, color='#e8edf5')

# Panel 1: distribución de ingreso (recorte en percentil 95 para mejor visualización)
cap = df['ing_cor'].quantile(0.95)
for label, subset, color in [
    ('No recibe', df[df['recibe_programa']==0], '#00c8ff'),
    ('Recibe',    df[df['recibe_programa']==1], '#00e5a0')
]:
    axes[0].hist(subset['ing_cor'].clip(upper=cap), bins=50,
                alpha=0.65, color=color, label=label, density=True)

axes[0].set_title('Distribución del ingreso trimestral', color='#e8edf5', fontsize=11)
axes[0].set_xlabel('Ingreso (pesos)', fontsize=9)
axes[0].set_ylabel('Densidad', fontsize=9)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Panel 2: gasto promedio por rubro
rubros = [c for c in ['alimentos','salud','educa_espa','transporte'] if c in df.columns]
etiquetas = {'alimentos':'Alimentos','salud':'Salud','educa_espa':'Educación','transporte':'Transporte'}
nombres = [etiquetas.get(r, r) for r in rubros]

vals_no = [df[df['recibe_programa']==0][r].mean() for r in rubros]
vals_si = [df[df['recibe_programa']==1][r].mean() for r in rubros]

x = range(len(rubros))
w = 0.35
axes[1].bar([i - w/2 for i in x], vals_no, w, label='No recibe', color='#00c8ff', alpha=0.85)
axes[1].bar([i + w/2 for i in x], vals_si, w, label='Recibe',    color='#00e5a0', alpha=0.85)
axes[1].set_title('Gasto promedio por rubro (trimestral)', color='#e8edf5', fontsize=11)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(nombres, fontsize=9)
axes[1].set_ylabel('Pesos promedio', fontsize=9)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y:,.0f}'))
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('figura1_perfil.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Figura guardada: figura1_perfil.png')

In [ ]:
# ── Figura 2: cobertura por entidad y tipo de localidad ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Cobertura de programas sociales — ENIGH 2022', fontsize=13, color='#e8edf5')

# Top 10 y bottom 10 entidades por cobertura
top10 = cobertura_entidad.sort_values('pct_cobertura', ascending=False).head(10)

bars = axes[0].barh(
    top10.index,
    top10['pct_cobertura'],
    color=['#00e5a0' if v > top10['pct_cobertura'].median() else '#7b61ff'
           for v in top10['pct_cobertura']],
    alpha=0.85
)
axes[0].set_title('Top 10 entidades por cobertura (%)', color='#e8edf5', fontsize=11)
axes[0].set_xlabel('% de hogares con programa', fontsize=9)
axes[0].grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars, top10['pct_cobertura']):
    axes[0].text(val + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=8, color='#e8edf5')

# Cobertura por tipo de localidad
cob_loc = df.groupby('tipo_loc')['recibe_programa'].mean() * 100
orden = ['Urbano grande (>100k)', 'Urbano medio (15-100k)',
         'Urbano pequeño (2.5-15k)', 'Rural (<2.5k)']
cob_loc = cob_loc.reindex([o for o in orden if o in cob_loc.index])

axes[1].bar(range(len(cob_loc)), cob_loc.values,
           color=['#00c8ff','#7b61ff','#ff6b35','#00e5a0'],
           alpha=0.85)
axes[1].set_title('Cobertura por tamaño de localidad', color='#e8edf5', fontsize=11)
axes[1].set_xticks(range(len(cob_loc)))
axes[1].set_xticklabels([l.split(' ')[0] + '\n' + ' '.join(l.split(' ')[1:])
                          for l in cob_loc.index], fontsize=8)
axes[1].set_ylabel('% de hogares con programa', fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')
for i, val in enumerate(cob_loc.values):
    axes[1].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontsize=9, color='#e8edf5')

plt.tight_layout()
plt.savefig('figura2_cobertura.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Figura guardada: figura2_cobertura.png')

---
## Sección 6 — Guardar resultados en PostgreSQL

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, dbname=PG_DATABASE,
    user=PG_USER, password=PG_PASSWORD, sslmode='require'
)
conn.autocommit = False
cur = conn.cursor()

# Activar PostGIS
cur.execute('CREATE EXTENSION IF NOT EXISTS postgis;')
conn.commit()

cur.execute('SELECT version();')
print('Conectado a PostgreSQL:', cur.fetchone()[0][:50])

In [ ]:
# Tabla: perfil resumido por entidad
cur.execute("""
    CREATE TABLE IF NOT EXISTS enigh_perfil_entidad (
        id              SERIAL PRIMARY KEY,
        entidad         INT,
        entidad_nom     TEXT,
        total_hogares   INT,
        pct_cobertura   FLOAT,
        ing_mediano     FLOAT,
        gasto_mediano   FLOAT,
        creado_en       TIMESTAMP DEFAULT NOW()
    )
""")
conn.commit()

# Tabla: beneficiarios con programa asignado
cur.execute("""
    CREATE TABLE IF NOT EXISTS enigh_beneficiarios (
        id              SERIAL PRIMARY KEY,
        folioviv        TEXT,
        foliohog        TEXT,
        entidad         INT,
        entidad_nom     TEXT,
        tipo_loc        TEXT,
        ing_cor         FLOAT,
        gasto_mon       FLOAT,
        bene_gob        FLOAT,
        programa        TEXT,
        creado_en       TIMESTAMP DEFAULT NOW()
    )
""")
conn.commit()
print('Tablas creadas.')

In [ ]:
# Insertar perfil por entidad
perfil_ent = df.groupby(['entidad','entidad_nom']).agg(
    total_hogares  = ('folioviv','count'),
    pct_cobertura  = ('recibe_programa','mean'),
    ing_mediano    = ('ing_cor','median'),
    gasto_mediano  = ('gasto_mon','median')
).reset_index()
perfil_ent['pct_cobertura'] = (perfil_ent['pct_cobertura'] * 100).round(2)

cur.executemany("""
    INSERT INTO enigh_perfil_entidad
        (entidad, entidad_nom, total_hogares, pct_cobertura, ing_mediano, gasto_mediano)
    VALUES (%s, %s, %s, %s, %s, %s)
""", [
    (int(r.entidad), r.entidad_nom, int(r.total_hogares),
     float(r.pct_cobertura), float(r.ing_mediano), float(r.gasto_mediano))
    for _, r in perfil_ent.iterrows()
])

# Insertar beneficiarios (muestra de 5,000 para no saturar)
muestra = df_beneficiarios.sample(min(5000, len(df_beneficiarios)), random_state=42)
cur.executemany("""
    INSERT INTO enigh_beneficiarios
        (folioviv, foliohog, entidad, entidad_nom, tipo_loc,
         ing_cor, gasto_mon, bene_gob, programa)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
""", [
    (str(r.folioviv), str(r.foliohog), int(r.entidad), str(r.entidad_nom),
     str(r.tipo_loc), float(r.ing_cor), float(r.gasto_mon),
     float(r.bene_gob), str(r.programa))
    for _, r in muestra.iterrows()
])
conn.commit()

print(f'Datos insertados en PostgreSQL:')
print(f'  enigh_perfil_entidad : {len(perfil_ent)} filas')
print(f'  enigh_beneficiarios  : {len(muestra)} filas')

In [ ]:
# Consulta: entidades con mayor cobertura y menor ingreso mediano
# (las más dependientes de programas sociales)
cur.execute("""
    SELECT
        entidad_nom,
        pct_cobertura,
        ROUND(ing_mediano::numeric, 0)   AS ing_mediano,
        ROUND(gasto_mediano::numeric, 0) AS gasto_mediano,
        total_hogares
    FROM enigh_perfil_entidad
    ORDER BY pct_cobertura DESC
    LIMIT 10
""")
cols = [d[0] for d in cur.description]
df_res = pd.DataFrame(cur.fetchall(), columns=cols)
print('Top 10 entidades por cobertura de programas sociales (ENIGH 2022):')
print(df_res.to_string(index=False))

---
## Sección 7 — Modelo de clasificación en Azure ML

Entrenamos un modelo que predice si un hogar recibe programas sociales
a partir de sus características socioeconómicas.  
Usamos Azure ML con el mismo flujo del Día 4 (Command Job + MLflow).

In [ ]:
# Preparar dataset para ML: features socioeconómicas → target: recibe_programa
features = [c for c in ['ing_cor','gasto_mon','alimentos','salud','educa_espa',
                         'transporte','bene_gob','entidad','tam_loc'] if c in df.columns]

df_ml = df[features + ['recibe_programa']].dropna()

# Subir dataset ML a Blob
ruta_ml = f'{BLOB_CONTAINER}/enigh_ml_dataset.csv'
with fs.open(ruta_ml, 'wb') as f:
    f.write(df_ml.to_csv(index=False).encode())

print(f'Dataset ML subido a Blob: {ruta_ml}')
print(f'  {len(df_ml):,} hogares | {len(features)} features | target: recibe_programa')
print(f'  Positivos: {df_ml["recibe_programa"].mean()*100:.1f}%')

In [ ]:
import os
os.makedirs('./src_dia5', exist_ok=True)

In [ ]:
%%writefile src_dia5/main.py
import os, argparse, pandas as pd, mlflow, mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data',                  type=str)
    parser.add_argument('--n_estimators',          type=int,   default=100)
    parser.add_argument('--max_depth',             type=int,   default=8)
    parser.add_argument('--registered_model_name', type=str,   default='enigh_programa_classifier')
    args = parser.parse_args()

    mlflow.start_run()
    mlflow.sklearn.autolog()

    df = pd.read_csv(args.data)
    print(f'Dataset: {df.shape[0]:,} filas, {df.shape[1]} columnas')

    X = df.drop(columns=['recibe_programa'])
    y = df['recibe_programa']

    mlflow.log_metric('pct_positivos', float(y.mean()))
    mlflow.log_metric('n_hogares', len(df))

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    clf = RandomForestClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        class_weight='balanced',
        random_state=42
    )
    clf.fit(X_train, y_train)

    y_pred  = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    mlflow.log_metric('roc_auc', auc)
    print(f'ROC-AUC: {auc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['No recibe','Recibe']))

    # Feature importance
    importances = pd.Series(clf.feature_importances_, index=X.columns)
    print('\nTop features:')
    print(importances.sort_values(ascending=False).head(5))

    mlflow.sklearn.log_model(
        sk_model=clf,
        registered_model_name=args.registered_model_name,
        artifact_path=args.registered_model_name
    )
    mlflow.end_run()
    print('Job completado.')

if __name__ == '__main__':
    main()

In [ ]:
from azure.ai.ml import MLClient, command, Input
from azure.identity import DefaultAzureCredential

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id     = AML_SUBSCRIPTION,
    resource_group_name = AML_RESOURCE_GROUP,
    workspace_name      = AML_WORKSPACE
)

# URL del dataset en Blob (abfss)
data_url = (f'azureml://subscriptions/{AML_SUBSCRIPTION}'
            f'/resourcegroups/{AML_RESOURCE_GROUP}'
            f'/workspaces/{AML_WORKSPACE}'
            f'/datastores/workspaceblobstore/paths/enigh_ml_dataset.csv')

job = command(
    code         = './src_dia5/',
    command      = ('python main.py '
                   '--data ${{inputs.data}} '
                   '--n_estimators ${{inputs.n_estimators}} '
                   '--max_depth ${{inputs.max_depth}}'),
    inputs       = dict(data=Input(type='uri_file', path=ruta_ml), n_estimators=150, max_depth=10),
    environment  = 'azureml://registries/azureml/environments/sklearn-1.5/labels/latest',
    compute      = 'cluster-dia5',
    display_name = 'enigh_programa_clasificacion',
    experiment_name = 'curso_azure_dia5_enigh'
)

returned_job = ml_client.create_or_update(job)
print(f'Job lanzado: {returned_job.name}')
print('Sigue el progreso en Azure ML Studio > Jobs > curso_azure_dia5_enigh')

In [ ]:
# Esperar resultado (5-10 min)
ml_client.jobs.stream(returned_job.name)
job_final = ml_client.jobs.get(returned_job.name)
print(f'Estado final: {job_final.status}')

---
## Sección 8 — Cierre del curso

### Lo que construiste esta semana

| Día | Servicio | Lo que hiciste |
|-----|----------|----------------|
| 1 | Portal Azure | Resource Group, exploración de servicios |
| 2 | Azure VM | Crear, configurar y conectarse via SSH |
| 3 | Blob Storage + PostgreSQL + PostGIS | Pipeline de datos, consultas geoespaciales |
| 4 | Databricks + Azure ML | Delta Lake, MLflow, Command Job con crédito bancario |
| 5 | Stack completo | ENIGH 2022 → Blob → PostgreSQL → Azure ML |

### Buenas prácticas para llevar

1. **Nunca hardcodees credenciales** en código que suba a Git. Usa `.env` + `.gitignore` o Azure Key Vault.
2. **Etiqueta todos tus recursos** (`az tag`) con proyecto, equipo y ambiente. Facilita el control de costos.
3. **Usa min_instances=0** en clusters de ML. Un cluster inactivo con 2 nodos puede costarte ~$150 USD/mes.
4. **Elimina recursos al terminar**: `az group delete --name <RG> --yes` borra todo lo del día.
5. **Versiona tus datos** con Delta Lake o al menos con fechas en el nombre del archivo en Blob.

### ¿Qué sigue?

- **Azure Synapse Analytics** — data warehouse + Spark + SQL serverless para análisis a escala institucional
- **Azure Key Vault** — gestión de secretos y credenciales para producción
- **Azure DevOps / GitHub Actions** — automatizar pipelines de datos con CI/CD
- **Azure Active Directory + RBAC** — control de acceso para equipos y proyectos de gobierno

In [ ]:
# Cerrar conexión a PostgreSQL
try:
    cur.close()
    conn.close()
    print('Conexión a PostgreSQL cerrada.')
except:
    pass

print()
print('Para eliminar TODOS los recursos del día 5 (libera costos):')
print(f'  az group delete --name TU_RESOURCE_GROUP --yes --no-wait')